# SEA TikTok Opportunity Radar — research walkthrough

**Synthetic replay by default. No market findings or actual sales are claimed.**

This notebook explains data availability, matched counter changes, purged chronological model evaluation, and a reproducible visual report. Production collection runs in GitHub Actions, not this notebook.

[中文说明](https://github.com/xuanbomalay/sea-tiktok-opportunity-radar/blob/main/docs/中文学习说明.md) | [Data Card](https://github.com/xuanbomalay/sea-tiktok-opportunity-radar/blob/main/docs/DATA_CARD.md)

Use a CPU session. No API keys are needed for this walkthrough.

## 1. Install the same research package

On Colab this checks out the public repository and installs its locked environment. Local QA can set `RADAR_REPO_ROOT`, `RADAR_SKIP_INSTALL=1`, and `RADAR_HEADLESS=1` to use an already installed checkout.

In [ ]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
REPO = Path(os.environ.get("RADAR_REPO_ROOT", "/content/sea-tiktok-opportunity-radar"))
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/xuanbomalay/sea-tiktok-opportunity-radar.git", str(REPO)], check=True)
if os.environ.get("RADAR_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.lock")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
os.chdir(REPO)
print("Python:", sys.version.split()[0])
print("Repository:", REPO)


## 2. Generate explicit synthetic history and replay

The generator fixes the random seed and creates invented products. Synthetic and live state directories cannot be mixed. This validates the pipeline mechanics; it does not validate demand or forecasting ability in a real market.

PNG export is skipped in this notebook so an extra browser installation is unnecessary. The deployed application exports PNGs with Kaleido/Chrome.

In [ ]:
from scripts.make_demo import demo_envelopes
from radar.storage import Store
WORK = REPO / "artifacts" / "notebook-replay"
WORK.mkdir(parents=True, exist_ok=True)
fixture = WORK / "synthetic.json"
fixture.write_text(json.dumps(demo_envelopes(), ensure_ascii=False), encoding="utf-8")
subprocess.run([sys.executable, "-m", "radar", "replay", str(fixture), "--synthetic", "--state-dir", str(WORK / "state"), "--output", str(WORK / "site"), "--no-agent", "--no-images"], check=True)
store = Store(WORK / "state")
records = store.observations()
print("Synthetic observations:", len(records))
print("Input SHA256:", hashlib.sha256(fixture.read_bytes()).hexdigest())


## 3. Inspect public-counter features

An observation formed at t can use only measurements available at t. The target at t is not mature until an exact observation at t+7 exists. Rounded displays, missing dates and counter resets are not filled with zero.

The same product/provider key must match at each required calendar date.

In [ ]:
import pandas as pd
from radar.modeling import training_frame, split_mature
frame = training_frame(records)
columns = ["date", "country", "niche", "product_id", "sold_precision", "increment7", "previous7", "target", "target_end"]
print(frame[columns].tail(12).to_string(index=False))


## 4. Verify chronological maturity purges

Dates, not randomly shuffled rows, define the splits. Label end must precede the next split. Preprocessing is trained on training data only. Never select a model using the test result.

In [ ]:
train, validation, test = split_mature(frame)
assert train.target_end.max() < validation.day.min()
assert validation.target_end.max() < test.day.min()
assert not set(train.date) & set(validation.date)
assert not set(validation.date) & set(test.date)
print("MATURITY PURGE PASS")
print({"train": len(train), "validation": len(validation), "test": len(test)})


## 5. Compare baseline, Ridge and gradient boosting

The baseline carries forward the recent seven-day increment. Model selection uses validation MAE. Test MAE and RMSE are reported for every model, including failed experiments. These synthetic errors are not evidence of real-world predictive skill.

In [ ]:
evaluation = json.loads((WORK / "state" / "evaluation.json").read_text(encoding="utf-8"))
assert evaluation["state"] == "evaluated" and evaluation["synthetic"]
assert evaluation["selected_model"] == min(evaluation["validation_mae"], key=evaluation["validation_mae"].get)
print("Selected on validation:", evaluation["selected_model"])
print(pd.DataFrame(evaluation["test_metrics"]).T.to_string())
print("Dataset SHA256:", evaluation["dataset_sha256"])


## 6. Visualize the sampled signals

The opportunity map compares matched demand growth and observed seller-to-listing ratio. The heatmap shows sample coverage, not market size. The trend chart uses changing search samples; it is not a daily orders chart. Prices are never directly ranked across currencies.

In [ ]:
from radar.render import figures
public = json.loads((WORK / "site" / "data" / "latest.json").read_text(encoding="utf-8"))
assert public["synthetic"] and public["data_mode"] == "synthetic_demo"
for name, figure in figures(public).items():
    if os.environ.get("RADAR_HEADLESS") != "1":
        figure.show()
    print("Figure:", name)
print("NOTEBOOK REPLAY PASS (synthetic fixtures only)")


## 7. Personal hypotheses and single-variable experiments

Before viewing a new test window, register one hypothesis, the expected mechanism, one changed variable, and a success metric. Examples: add a lagged rating feature, remove price, or compare localized search variants. Preserve the baseline, record negative results and do not optimize on the final test period.

Real-data evaluation must independently satisfy 84 observation dates and 500 mature labels. A small daily sample may take longer or fail to form a usable panel. Data collection success, forecast validity, efficiency gains and business sales are separate acceptance milestones.

The bounded Agent is tested separately. It can query computed metrics and select evidence IDs; it cannot invent market statistics. Without a free-tier API key the daily brief is explicitly rule-based.